# What this simulation is doing

This notebook runs a **conditioned 2D Navier-Stokes smoke** system (`ConditionedNavierStokes2D`).

- **State variables**: `smoke` (scalar), `u` (x-velocity), `v` (y-velocity).
- **Conditioning variable**: `buoyancy_y` controls vertical coupling from smoke to momentum.
- **Dynamics**: semi-Lagrangian advection + viscosity + buoyancy forcing + pressure projection.
- **Boundary conditions**: periodic in both spatial directions.

### How to read the three plotted channels
- **Smoke fades toward homogeneity** because there is no source term; diffusion (and some numerical diffusion from semi-Lagrangian advection) smooths gradients over long times.
- **Velocity starts at zero by design** (`u=v=0` initial condition), then buoyancy spins it up from smoke anomalies.
- **Velocity can stay active longer** while smoke still has residual structure; once smoke becomes nearly uniform, buoyancy weakens and viscosity damps flow.
- **Channel colors are independently normalized**, so apparent contrast in `smoke` vs `u`/`v` is not a direct magnitude comparison between channels.

This mirrors the PDEArena *conditioned* Navier-Stokes setup where trajectories are conditioned on buoyancy strength.

## Governing equations (conditioned smoke-flow)

We evolve an incompressible velocity field $\mathbf{u}=(u,v)$ and a smoke scalar $s$:
$$
\frac{\partial s}{\partial t} + \mathbf{u}\cdot\nabla s = \kappa \nabla^2 s,
$$
$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u}\cdot\nabla)\mathbf{u}
= -\nabla p + \nu \nabla^2 \mathbf{u} + (0, \beta (s-\bar{s})),
\qquad \nabla\cdot\mathbf{u}=0.
$$

### Variable definitions

- $s(x,y,t)$: smoke/passive scalar field.
- $\mathbf{u}(x,y,t)=(u,v)$: velocity field.
- $p(x,y,t)$: pressure-like projection potential.
- $\nu$: kinematic viscosity.
- $\kappa$: smoke diffusivity.
- $\beta$: buoyancy coefficient (this is the conditioning parameter `buoyancy_y`).

In PDEArena's conditioned dataset generation, different trajectories use different fixed values of buoyancy strength.

In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import ConditionedNavierStokes2D as Sim
from autosim.utils import plot_spatiotemporal_video


def to_pdearena_api(batch, skip_frames=0):
    """Convert autosim batch -> PDArena-like dict."""
    params = batch["constant_scalars"]
    data = batch["data"]

    if skip_frames > 0:
        data = data[:, 1 + skip_frames :, ...]

    return {
        "params": params,
        "outputs": {
            "u": data[..., 0],
            "vx": data[..., 1],
            "vy": data[..., 2],
        },
    }


In [ ]:
# Random field example
sim = Sim(
    return_timeseries=True,
    log_level="warning",
    n=64,
    T=320.0,
    dt=0.1,
    nu=0.03,
    L=32.0,
    cfl=0.35,
    snapshot_dt=1.0,
    parameters_range={"buoyancy_y": (0.25, 0.45)},
)

batch_original = sim.forward_samples_spatiotemporal(n=2, random_seed=42)
params_original = batch_original["constant_scalars"]
outputs_original = batch_original["data"]


In [ ]:
print("constant_scalars shape:", params_original.shape)
print("data shape:", outputs_original.shape)
print("sample params (trajectory 0):", {"buoyancy_y": float(params_original[0, 0])})

In [ ]:
anim = plot_spatiotemporal_video(
    outputs_original,
    batch_idx=0,
    channel_names=["u", "vx", "vy"],
)

HTML(anim.to_jshtml())


## PDEArena-style run with navierstokes2dsmoke-like settings

This section uses the common PDEArena-like configuration values:

- `n=128`, `L=32`, `nu=0.01`
- `nt=56`, `skip_nt=8`, `sample_rate=1`
- `dt=(102-18)/56=1.5`
- fixed `buoyancy_y=0.5`
- outputs follow PDArena convention by dropping `t=0` and warmup frames.

Two new optional flags bring behaviour closer to PDEArena:

| Flag | `'periodic'` (default) | `'neumann'` (PDEArena-like) |
|---|---|---|
| `bc_mode` | wraps fields at edges | zero-gradient smoke, no-slip velocity |

| Flag | `'anomaly'` (default) | `'raw'` (PDEArena-like) |
|---|---|---|
| `buoyancy_mode` | forces on `smoke - mean(smoke)` | forces on raw `smoke` |

In [ ]:
nt = 56
# nt = 112
skip_nt = 8
sample_rate = 1
dt_pde = (102.0 - 18.0) / nt  # 1.5
T_total_pde = (nt + skip_nt) * dt_pde

sim_pde = Sim(
    return_timeseries=True,
    log_level="warning",
    # n=128,
    n=64,
    T=T_total_pde,
    dt=dt_pde,
    nu=0.01,
    L=32.0,
    cfl=0.35,
    noise_scale=11.0,
    smooth_steps=6,
    # --- PDEArena-alignment flags ---
    # bc_mode="periodic",    # swap to "neumann" for bounded domain (PDEArena BCs)
    # buoyancy_mode="anomaly",  # swap to "raw" for PDEArena-exact buoyancy
    bc_mode="neumann",    # swap to "neumann" for bounded domain (PDEArena BCs)
    buoyancy_mode="raw",  # swap to "raw" for PDEArena-exact buoyancy
    snapshot_dt=dt_pde * sample_rate,
    parameters_range={"buoyancy_y": (0.5, 0.5)},
    smoke_diffusivity=0.0
)

batch_pde = sim_pde.forward_samples_spatiotemporal(n=1, random_seed=42)
pdearena_pde = to_pdearena_api(batch_pde, skip_frames=skip_nt)

print("params shape:", pdearena_pde["params"].shape)
print("outputs['u'] shape:", pdearena_pde["outputs"]["u"].shape)
print("sample params:", {"buoyancy_y": float(pdearena_pde["params"][0, 0])})

In [ ]:
anim_pde = plot_spatiotemporal_video(
    batch_pde["data"][:, 1 + skip_nt :, ...],
    batch_idx=0,
    channel_names=["u", "vx", "vy"],
)

HTML(anim_pde.to_jshtml())
